In [13]:
import os
import cv2
import torch
import numpy as np
import random
from PIL import Image
from tqdm import tqdm
from torchvision.transforms import ToTensor

class preprocessing_baseline:
    def __init__(self, threshold=128):
        self.threshold = threshold

    def __call__(self, img):  # img: PIL.Image
        img_np = np.array(img.convert("RGB"))
        resized = cv2.resize(img_np, (64, 64))
        gray = cv2.cvtColor(resized, cv2.COLOR_RGB2GRAY)
        _, binarized = cv2.threshold(gray, self.threshold, 255, cv2.THRESH_BINARY)
        
        tensor = ToTensor()(Image.fromarray(binarized)) * 255  # [1, H, W], byte range
        return tensor.byte()
        
class preprocessing_ours:
    def __init__(self, size=64, threshold=128, crop=2):
        self.size = size
        self.threshold = threshold
        self.crop = crop

    def __call__(self, img):  # img: PIL.Image
        img_np = np.array(img.convert("RGB"))
        if self.crop > 0:
            h, w, _ = img_np.shape
            img_np = img_np[self.crop:h-self.crop, self.crop:w-self.crop, :]

        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        log_edge = np.abs(cv2.Laplacian(gray, cv2.CV_64F, ksize=3))
        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel_edge = np.sqrt(sobelx**2 + sobely**2)

        log_norm = cv2.normalize(log_edge, None, 0, 1.0, cv2.NORM_MINMAX)
        sobel_norm = cv2.normalize(sobel_edge, None, 0, 1.0, cv2.NORM_MINMAX)
        w1 = np.mean(log_norm)
        w2 = np.mean(sobel_norm)
        edge = (w1 * log_norm + w2 * sobel_norm)
        edge = cv2.normalize(edge, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        blurred = cv2.GaussianBlur(edge, (3, 3), 1)
        resized = cv2.resize(blurred, (self.size, self.size))
        _, binarized = cv2.threshold(resized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        tensor = ToTensor()(Image.fromarray(binarized)) * 255  # shape: [1, H, W], values: 0~255
        return tensor.byte()

class AddRandomNoise:
    def __init__(self, candidate_ratios=[0.05, 0.10, 0.25, 0.50]):
        self.candidate_ratios = candidate_ratios

    def __call__(self, tensor_img):  # tensor: [1, H, W], byte (0 or 255)
        noise_ratio = random.choice(self.candidate_ratios)
        img = tensor_img.clone()
        h, w = img.shape[1], img.shape[2]
        num_noisy = int(h * w * noise_ratio)
        if num_noisy > 0:
            coords = torch.randperm(h * w)[:num_noisy]
            y = coords // w
            x = coords % w
            img[0, y, x] = 255 - img[0, y, x]
        return img

class AddNoise:
    def __init__(self, noise_ratio):
        self.noise_ratio = noise_ratio

    def __call__(self, img):
        assert isinstance(img, np.ndarray) 
        img_np = img.copy()
        h, w = img_np.shape
        num_pixels = h * w
        num_noisy = int(num_pixels * self.noise_ratio)
        coords = np.random.choice(num_pixels, num_noisy, replace=False)
        y, x = np.unravel_index(coords, (h, w))
        img_np[y, x] = 255 - img_np[y, x]
        return Image.fromarray(img_np.astype(np.uint8))

def mismatch(tensor_img, radius=1):
    h, w = tensor_img.shape
    output = tensor_img.clone()
    for i in range(radius, h - radius):
        for j in range(radius, w - radius):
            patch = tensor_img[i-radius:i+radius+1, j-radius:j+radius+1]
            center = tensor_img[i, j]
            neighbors = torch.cat((patch.flatten()[:4], patch.flatten()[5:]))
            if torch.all(neighbors != center):
                output[i, j] = 255 - center
    return output

def diagonal_solo(tensor_img):
    h, w = tensor_img.shape
    output = tensor_img.clone()
    for i in range(1, h-1):
        for j in range(1, w-1):
            center = tensor_img[i, j]
            diag = [tensor_img[i-1, j-1], tensor_img[i-1, j+1],
                    tensor_img[i+1, j-1], tensor_img[i+1, j+1]]
            straight = [tensor_img[i-1, j], tensor_img[i+1, j],
                        tensor_img[i, j-1], tensor_img[i, j+1]]
            if sum([d == center for d in diag]) == 1 and sum([s != center for s in straight]) == 4:
                output[i, j] = 255 - center
    return output

def train(model, device, trainloader, optimizer, criterion, num_epochs, binarizer, save_path='./prob2_2_ours_weight'):
    os.makedirs(save_path, exist_ok=True)
    history = np.zeros((0, 3))

    noise_adder = AddRandomNoise()

    for epoch in tqdm(range(num_epochs)):
        model.train()
        total_loss, correct, total = 0, 0, 0

        for images, labels in trainloader:
            batch = []
            for img in images:
                img = binarizer(img)                 
                img = noise_adder(img)               # Add noise
                img = img.to(device)[0]              # shape: [H, W]
                img = mismatch(img)                  # mismatch
                img = diagonal_solo(img)             # diagonal solo
                batch.append(img)

            X = torch.stack(batch).unsqueeze(1).float()  # [B,1,H,W]
            y = labels.to(device)

            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

        avg_loss = total_loss / len(trainloader)
        avg_acc = correct / total
        history = np.vstack((history, [epoch+1, avg_loss, avg_acc]))
        print(f"Epoch {epoch+1}: Loss {avg_loss:.4f}, Accuracy {avg_acc:.4f}")

        if (epoch+1) % 10 == 0:
            savefile = os.path.join(save_path, f"weight{epoch+1}.pth")
            torch.save(model.state_dict(), savefile)
            print(f"Checkpoint saved: {savefile}")

    return history

In [15]:
import torchvision.models as models
import torch
import torch.nn as nn
from torchvision import datasets
from torch.utils.data import DataLoader

def pil_collate_fn(batch):
    images, labels = zip(*batch)  # batch: list of (PIL.Image, label)
    return list(images), torch.tensor(labels)

trainset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Train')  # transform 생략
testset = datasets.ImageFolder('/home/dh/venv/dataset/Animals/Test')  # transform 생략
trainloader = DataLoader(trainset, batch_size=40, shuffle=True, num_workers=2, drop_last=True, collate_fn=pil_collate_fn)
testloader  = DataLoader(testset,  batch_size=40, shuffle=True, num_workers=2, drop_last=True, collate_fn=pil_collate_fn)

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x74cc785fe980>
Traceback (most recent call last):
  File "/home/dh/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/home/dh/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1582, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.12/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 1136, in wait
    ready = selector.select(timeout)
            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/selectors.py", line 415, in select
    fd_event_list = self._selector.poll(timeout

KeyboardInterrupt: 

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
prob2_1 = models.resnet18()
prob2_1.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
num_ftrs = prob2_1.fc.in_features
prob2_1.fc = nn.Linear(num_ftrs, 4)  
prob2_1 = prob2_1.to(device)

# 학습 설정
num_epochs = 50
lr = 0.001
optimizer = torch.optim.Adam(prob2_1.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

# 전처리 방식 선택 (OURS 또는 Baseline 중)
#binarizer = Baseline(threshold=128)
binarizer = OURS()

# 학습 실행
history = train(prob2_1, device, trainloader, optimizer, criterion, num_epochs, binarizer)

In [8]:
def test(model, device, test_loader, criterion, binarizer, noise_adder):
    test_loss = []
    test_accuracy = []
    model.eval()

    with torch.no_grad():
        for images, labels in tqdm(test_loader):
            batch = []
            for img in images:
                img = binarizer(img)         # PIL → tensor [1, H, W]
                img = img[0].cpu().numpy()   # → numpy [H, W] for AddNoise
                img = noise_adder(img)       # AddNoise: numpy → PIL
                img = binarizer(img)         # 다시 binarize (PIL → tensor)
                img = mismatch(img.to(device)[0])
                img = diagonal_solo(img)
                batch.append(img)

            X = torch.stack(batch).unsqueeze(1).float()
            y = labels.to(device)

            output = model(X)
            loss = criterion(output, y)

            pred = output.argmax(dim=1)
            accuracy = (pred == y).float().mean().item()

            test_loss.append(loss.item())
            test_accuracy.append(accuracy)

    avg_loss = sum(test_loss) / len(test_loss)
    avg_accuracy = sum(test_accuracy) / len(test_accuracy)
    print(f"Test Loss: {avg_loss:.4f} | Accuracy: {avg_accuracy:.4f}")

In [20]:
import torchvision.models as models
import torch.nn as nn
import torch

binarizer = OURS()
noise_adder = AddNoise(noise_ratio=0.1)

# 저장된 모델 파일들
model_paths = [
    './prob2_component_weight/weight10.pth',
    './prob2_component_weight/weight20.pth',
    './prob2_component_weight/weight30.pth',
    './prob2_component_weight/weight40.pth',
    './prob2_component_weight/weight50.pth',
]

# 모델 테스트 반복
for path in model_paths:
    print(f"\n Testing {path}")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = models.resnet18()
    num_ftrs = model.fc.in_features
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    prob2_1.fc = nn.Sequential(
    nn.Linear(num_ftrs, 4))
    model.load_state_dict(torch.load(path, map_location=device))
    model = model.to(device)

    test(model, device, testloader, criterion, binarizer, noise_adder)

NameError: name 'AddNoise' is not defined